# DotMatch Runnable Demo

This notebook runs a deterministic CRISPR guide-counting smoke demo using the same small target and FASTQ shapes exercised by the CLI tests. It writes outputs to a temporary directory so the repository stays clean.

In [ ]:
from pathlib import Path
import csv
import json
import importlib.util
import os
import subprocess
import sys
import tempfile
import urllib.request

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "demo-data" / "crispr_guides.tsv").exists():
            return path
    if importlib.util.find_spec("dotmatch") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "dotmatch==0.2.2"], check=True)
    root = Path(tempfile.mkdtemp(prefix="dotmatch-colab-"))
    for relative in ("demo-data/crispr_guides.tsv", "demo-data/reads.fastq"):
        target = root / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        url = "https://raw.githubusercontent.com/dnncha/dotmatch/main/" + relative
        urllib.request.urlretrieve(url, target)
    print("Downloaded the small synthetic fixture from the public repository.")
    return root

root = find_repo_root(Path.cwd())
targets = root / "demo-data" / "crispr_guides.tsv"
reads = root / "demo-data" / "reads.fastq"

assert targets.exists(), targets
assert reads.exists(), reads
print(targets)
print(reads)

In [ ]:
with tempfile.TemporaryDirectory(prefix="dotmatch-demo-") as tmp:
    out_dir = Path(tmp)
    counts = out_dir / "counts.tsv"
    assignments = out_dir / "assignments.tsv"
    summary = out_dir / "summary.json"

    cmd = [
        sys.executable,
        "-m",
        "dotmatch.cli",
        "count",
        "--targets",
        str(targets),
        "--reads",
        str(reads),
        "--target-start",
        "0",
        "--target-length",
        "4",
        "--k",
        "1",
        "--out",
        str(counts),
        "--assignments",
        str(assignments),
        "--summary",
        str(summary),
    ]
    env = {**os.environ, "DOTMATCH_PYTHON_NO_DELEGATE": "1"}
    subprocess.run(cmd, check=True, env=env)

    with counts.open(encoding="utf-8") as fh:
        counts_rows = list(csv.DictReader(fh, delimiter="\t"))
    with assignments.open(encoding="utf-8") as fh:
        assignments_rows = list(csv.DictReader(fh, delimiter="\t"))
    summary_data = json.loads(summary.read_text(encoding="utf-8"))

counts_rows

In [ ]:
assignments_rows

In [ ]:
summary_data

In [ ]:
assert summary_data["total_reads"] == 3
assert summary_data["assigned_unique"] == 2
assert summary_data["unmatched"] == 1
guide_1 = next(row for row in counts_rows if row["target_id"] == "guide_1")
assert int(guide_1["count_total"]) == 2
print("DotMatch demo completed successfully.")